# Spatial Joins: Point-in-Polygon (Day 2)\n\nAssign coffee shops to neighborhoods, count per district.

In [ ]:
import geopandas as gpd\nfrom shapely.geometry import Point, Polygon\nimport matplotlib.pyplot as plt

In [ ]:
# Sample coffee shops (points)\ncoffee_data = [\n    {'name': 'Cafe A', 'coords': (1, 1)},\n    {'name': 'Cafe B', 'coords': (2, 3)},\n    {'name': 'Cafe C', 'coords': (4, 2)},\n    {'name': 'Cafe D', 'coords': (3, 4)},\n    {'name': 'Cafe E', 'coords': (5, 1)},\n]\n\npoints = [Point(x, y) for d in coffee_data for x, y in [d['coords']]]\ncafes_gdf = gpd.GeoDataFrame(coffee_data, geometry=points, crs='EPSG:4326')\ncafes_gdf

In [ ]:
# Sample neighborhoods (polygons)\nneighborhoods_data = [\n    {'name': 'North District', 'bounds': [(0,0), (3,0), (3,3), (0,3)]},\n    {'name': 'South District', 'bounds': [(3,0), (6,0), (6,3), (3,3)]},\n    {'name': 'East District', 'bounds': [(3,3), (6,6), (3,6), (3,3)]},\n]\n\npolys = [Polygon(bounds) for data in neighborhoods_data for bounds in [data['bounds']]]\nhoods_gdf = gpd.GeoDataFrame(neighborhoods_data, geometry=polys, crs='EPSG:4326')\nhoods_gdf

In [ ]:
# Spatial join: points within polygons\njoined = gpd.sjoin(cafes_gdf, hoods_gdf, how='left', predicate='within')\nprint(joined)\n\n# Count shops per neighborhood\ncounts = joined.groupby('name_right')['name_left'].count().reset_index(name='shop_count')\nhoods_gdf = hoods_gdf.merge(counts, on='name', how='left').fillna(0)\nhoods_gdf

In [ ]:
# Visualize\nfig, ax = plt.subplots(1, 2, figsize=(12,5))\nhoods_gdf.plot(column='shop_count', legend=True, ax=ax[0], cmap='OrRd')\ncafes_gdf.plot(ax=ax[1], color='red', markersize=50)\nhoods_gdf.boundary.plot(ax=ax[1], color='black')\nplt.show()

In [ ]:
# Export GeoJSON\nhoods_gdf.to_file('neighborhoods_shops.geojson', driver='GeoJSON')\nprint('Saved neighborhoods_shops.geojson – view on geojson.io')